# OpenAI Responses API

## What is the OpenAI Responses API?

The Responses API is a new API released in March 2025. It is a combination of the traditional 
Chat Completions API and the Assistants API, providing support for:

- **Traditional Chat Completions:** Facilitates seamless conversational AI experiences.
- **Web Search:** Enables real-time information retrieval from the internet.
- **File Search:** Allows searching within files for relevant data.

Accordingly, the Assistants API will be retired in 2026. 

> **For new users, OpenAI recommends using the Responses API instead of the Chat Completions API to leverage its expanded capabilities.**

For a comprehensive comparison between the Responses API and the Chat Completions API, refer to the official OpenAI documentation: 
[Responses vs. Chat Completions](https://platform.openai.com/docs/guides/responses-vs-chat-completions).

## Summary of This Notebook
This notebook provides a hands-on guide for using the **OpenAI Responses API** to analyze tweets. 
It covers essential techniques such as:

- **Creating a vector store** and uploading tweets for semantic search.
- **Using file search** to analyze private datasets.
- **Performing a web search** to retrieve the latest public information.
- **Utilizing stateful responses** to maintain conversation context.
- **Combining file and web search** to enhance retrieval-augmented generation (RAG) applications.

By the end of this notebook, users will be able to integrate OpenAI's Responses API for efficient data retrieval and analysis of structured and unstructured data.

## Install Required Libraries
To use the OpenAI Responses API, we need to install the following libraries:

- **`openai`**: Provides access to OpenAI's APIs, including the Responses API

In [1]:
pip install openai -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sparkmagic 0.21.0 requires pandas<2.0.0,>=0.17.1, but you have pandas 2.3.3 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


## Import Required Libraries

In [2]:
from IPython.display import Markdown, display
import boto3
from botocore.exceptions import ClientError
import json
import io

## Retrieve Secrets from AWS Secrets Manager

In [3]:
def get_secret(secret_name):
    region_name = "us-east-1"

    # Create a Secrets Manager client
    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        raise e

    secret = get_secret_value_response['SecretString']
    
    return json.loads(secret)

## Initialize OpenAI Client

In [4]:
from openai import OpenAI
openai_api_key  = get_secret('openai')['api_key']

client = OpenAI(api_key=openai_api_key)

## File Search API

### Introduction to File Search
File search API enables efficient retrieval of relevant information 
from uploaded files by leveraging vector-based indexing. This feature is particularly useful 
for searching large datasets, extracting insights, and improving retrieval-augmented generation (RAG) applications.

Unlike traditional keyword-based searches, the Responses API uses embeddings 
to identify semantically relevant content, making it ideal for analyzing structured 
and unstructured text data (OpenAI, 2025).

For more details, visit the official OpenAI documentation: 
[File Search in Responses API](https://platform.openai.com/docs/guides/tools-file-search).

### Create a Vector Store

In [5]:
vector_store = client.vector_stores.create(
    name="my_vector_store"
)
vector_store_id = vector_store.id
print(vector_store_id)

vs_6912649333c88191823d9e10ce4db351


### Upload Files

In [6]:
with open('tweet_text.json', 'rb') as f:
    file = client.files.create(
        file=f,            # file-like object
        purpose="assistants"
    )

file_id = file.id
print(file_id)

file-6KAZACsCNZhjrk4wdsjfYc


### Attach File to Vector Store

In [7]:
attach_status =client.vector_stores.files.create(
    vector_store_id=vector_store_id,
    file_id=file_id
            )

print(attach_status.id)

file-6KAZACsCNZhjrk4wdsjfYc


### Query the Vector Store

In [8]:
query = "the latest development in generativeAI"

In [9]:
search_results = client.vector_stores.search(
    vector_store_id=vector_store_id,
    query=query
)

for result in search_results.data[:5]:
    print(result.content[0].text[:100] + '\n Relevant score: ' + str(result.score))

They have been VERY clear that Chatgpt, these AI videos…"
  }
},
{
  "_id": {
    "$oid": "68e56b1e1
 Relevant score: 0.6760340526672118
They have been VERY clear that Chatgpt, these AI videos…"
  }
},
{
  "_id": {
    "$oid": "68e56b1f1
 Relevant score: 0.6010846063542131
Learning Plans. Use coupon code 𝐒𝐏𝐋𝟑𝟎 at checkout.🎯\n\nJoin Now 👉 https://t.co/LS2JuCrVmz\n\n#AI #Ce
 Relevant score: 0.5984074448147717
Create stunning, cinematic videos from a prompt—now with audio, physics, and cameos. One prompt = en
 Relevant score: 0.591898195864025
They have been VERY clear that Chatgpt, these AI videos…"
  }
},
{
  "_id": {
    "$oid": "68e56b1d1
 Relevant score: 0.5630078302657743


## OpenAI Response API

### Simple Response

In [10]:
simple_response = client.responses.create(
  model="gpt-4o",
  input=[
      {
          "role": "user",
          "content": query
      }
  ]
)

In [11]:
display(Markdown(simple_response.output_text))

As of 2023, generative AI continues to advance rapidly with several noteworthy developments:

1. **Multimodal Models**: Models like GPT-4 and beyond have improved at integrating text, image, and audio data, enabling more sophisticated capabilities across different media types.

2. **Real-time Interaction**: Enhanced real-time processing allows for more seamless human-AI interactions, including live translation and dynamic content generation.

3. **Fine-tuning and Customization**: More accessible tools for fine-tuning pre-trained models on specific tasks have been developed, allowing businesses to create bespoke AI solutions more easily.

4. **Ethical Enhancements**: There's a greater focus on embedding ethical guidelines within AI systems to prevent bias and ensure more responsible use.

5. **AI Creativity**: Advances in creativity-oriented AI have led to more compelling content generation in music, art, and storytelling, pushing the boundaries of automated creativity.

6. **Efficiency Improvements**: New architectures and techniques have increased efficiency, reducing computational costs and making powerful generative AI accessible to more users.

7. **Regulatory and Safety Measures**: As generative AI becomes more powerful, there’s a strong emphasis on implementing regulatory frameworks and safety measures to manage its impact.

These trends signify ongoing innovation and raise important considerations for the future use of generative AI.

### File Search Response

In [12]:

file_search_response = client.responses.create(
    input= query,
    model="gpt-4o",
    temperature = 0,
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store_id],
    }]
)

In [13]:
display(Markdown(file_search_response.output_text))


The latest developments in generative AI include:

1. **Agentic Workflows**: Amazon Web Services is exploring the future of AI with a focus on agentic workflows, which are likely to enhance automation and efficiency in various applications.

2. **AI in Supply Chain**: Atos has developed an AI-powered Supply Chain Disruption Analysis tool using generative AI to assess risks and boost resilience.

3. **Market Growth**: The global generative AI market is projected to reach $1.18 billion this year, indicating rapid growth and adoption across industries.

4. **Creative Industries**: Generative AI is transforming creative industries by enabling the creation of content, designs, and strategies, reshaping how businesses innovate.

5. **AI in Media**: Hollywood has signed a landmark agreement on the use of AI in post-production, allowing digital doubles under certain conditions.

These developments highlight the expanding role of generative AI in various sectors, from supply chain management to media and creative industries.

## Web Search API

### Introduction to Web Search
The OpenAI Web Search tool allows models to retrieve real-time information from the internet. 
This capability is particularly useful for obtaining up-to-date data, fact-checking, and expanding knowledge 
without relying solely on pre-trained information. 

By leveraging OpenAI's web search functionality, the Responses API can fetch external data 
and provide accurate, relevant results in real time (OpenAI, 2025). 
This feature enhances applications that require the latest insights, such as news aggregation, research, 
or dynamic content generation.

For more details, visit the official OpenAI documentation: 
[Web Search in Responses API](https://platform.openai.com/docs/guides/tools-web-search).

### Perform Web Search

In [14]:
web_search_response = client.responses.create(
    model="gpt-4o",  # or another supported model
    input= query,
    tools=[
        {
            "type": "web_search"
        }
    ]
)

In [15]:
display(Markdown(web_search_response.output_text))

Below is a **thorough, structured overview** of the **latest developments in generative AI** as of **Monday, November 10, 2025**. These trends span foundational model releases, enterprise tools, hardware advancements, regulatory shifts, and creative integrations—with **explicit, up-to-date citations**.

---

##  Foundational Model Innovations

- **OpenAI’s GPT‑5**, a multimodal large language model, was released on **August 7, 2025**, offering integrated reasoning and non-reasoning capabilities across text, code, and image modalities. It is accessible via ChatGPT, Microsoft Copilot, and the OpenAI API. ([en.wikipedia.org](https://en.wikipedia.org/wiki/GPT-5?utm_source=openai))  
- **Alibaba’s Qwen series** continues to evolve:
  - The **Qwen3 family** launched on **April 28, 2025**, with models from 0.6B to 235B activated parameters, supporting up to a 128K token context window and reasoning functionalities. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Qwen?utm_source=openai))
  - **Qwen3-Max** debuted on **September 5, 2025**, surpassing competitors like Claude 4 Opus in non-reasoning benchmarks, with “thinking mode” activated publicly in early November 2025. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Qwen?utm_source=openai))

---

##  Visual & Multimedia Generative Advances

- **Nano Banana** (officially *Gemini 2.5 Flash Image*)—a text-to-image generative model with advanced editing, subject consistency, multi-image fusion, and SynthID watermarking—launched on **August 26, 2025**. It quickly went viral, driving over **10 million new Gemini app users** and exceeding **200 million image edits** within weeks. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Nano_Banana?utm_source=openai))  
- **Runway Gen‑4**, introduced on **March 31, 2025**, is a text-to-video model generating 5–10 second clips with improved scene consistency, camera motion, and style transfer from reference images. Outputs reach 720p at 24 fps; available via API and web interface. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Gen-4_%28AI_image_and_video_model%29?utm_source=openai))  
- **Kling AI**, from Kuaishou, advanced its text-to-video series with version **2.1 released on May 28, 2025**, featuring diffusion-based plus 3D VAE architecture for improved spatiotemporal quality. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Kling_AI?utm_source=openai))

---

##  Enterprise Deployments & Media Integrations

- **Meta’s Generative Ads Recommendation Model (GEM)**, unveiled **today (November 10, 2025)**, is an LLM‑scale foundation model powering ad recommendations across Facebook and Instagram. It enhances ad precision and advertiser ROI through architectural scalability across thousands of GPUs. ([engineering.fb.com](https://engineering.fb.com/2025/11/10/ml-applications/metas-generative-ads-model-gem-the-central-brain-accelerating-ads-recommendation-ai-innovation/?utm_source=openai))  
- **Adobe’s Firefly Boards**, now launched globally (last month), is an AI-first collaborative platform integrating generative video capabilities powered by Adobe, Google, Runway, Luma AI, and others. ([timesofindia.indiatimes.com](https://timesofindia.indiatimes.com/technology/artificial-intelligence/adobe-launches-firefly-boards-globally-with-new-ai-video-models-and-features/articleshow/124153220.cms?utm_source=openai))  
- **TIME AI Agent**—released today—combines generative language understanding, voice synthesis, translation, and search into a news‐interaction platform developed in collaboration with Scale AI, enabling summarization, audio report creation, and interactive exploration of TIME journalism. ([time.com](https://time.com/7332572/the-story-behind-the-time-ai-agent/?utm_source=openai))  
- **Business Insider** is starting to publish AI-authored stories under the byline “Business Insider AI,” in testing as of two weeks ago. Content will be edited by human staff and clearly labeled for transparency. ([nypost.com](https://nypost.com/2025/10/24/business/business-insider-to-start-publishing-stories-by-ai-author/?utm_source=openai))

---

##  Infrastructure & Hardware Momentum

- **Qualcomm** revealed its **AI200 and AI250 inference accelerators** two weeks ago. Designed to compete with AMD and Nvidia in data centers, AI200 launches in 2026 with 768 GB LPDDR, rack-scale deployment, and advanced cooling. AI250, arriving later, emphasizes near-memory compute and dynamic resource sharing. ([tomshardware.com](https://www.tomshardware.com/tech-industry/artificial-intelligence/qualcomm-unveils-ai200-and-ai250-ai-inference-accelerators-hexagon-takes-on-amd-and-nvidia-in-the-booming-data-center-realm?utm_source=openai))

---

##  Education, Security & Policy Context

- **Google’s Veteran AI Leader Program** launches **November 13–14, 2025**, offering generative AI training and certification to veterans transitioning into tech careers. ([startuphub.ai](https://www.startuphub.ai/ai-news/ai-research/2025/google-accelerates-veteran-ai-training-initiatives/?utm_source=openai))  
- **Security Concerns**: A report published today highlights that as generative AI becomes prevalent across customer service, product design, and R&D, enterprises are increasingly vulnerable to novel security threats—demanding fortified business defenses. ([webpronews.com](https://www.webpronews.com/genais-hidden-perils-fortifying-business-defenses-in-2025/?utm_source=openai))

---

##  Academic & Applied Generative AI Research

- A review from **August 7, 2025** surveys how generative models (GANs, VAEs, diffusion, multimodal architectures) are transforming medical imaging—enhancing synthesis, modality translation, diagnostic support, and treatment planning—while flagging challenges around hallucination, data privacy, generalization, and regulation. ([arxiv.org](https://arxiv.org/abs/2508.09177?utm_source=openai))  
- A **recent survey (October 23, 2025)** provides a deep taxonomy of generative models (GANs, VAEs, diffusion), outlining innovations in output quality, diversity, control, and ethical considerations across real-world applications. ([arxiv.org](https://arxiv.org/abs/2510.21887?utm_source=openai))

---

##  Summary Table

| Area                    | Key Trend / Release                        | Date            |
|-------------------------|---------------------------------------------|-----------------|
| Foundation Models       | GPT‑5, Qwen3-Max, Qwen3 family             | Aug–Nov 2025    |
| Visual Generative Models| Nano Banana, Runway Gen‑4, Kling AI        | Mar–Aug 2025    |
| Enterprise Integration  | Meta GEM, Adobe Firefly Boards, TIME Agent, Business Insider AI | Nov 2025         |
| Hardware                | Qualcomm AI200 & AI250 accelerators        | 2026–2027 (announced Nov 2025) |
| Education & Security    | Google veteran certification; enterprise security risks | Nov 2025        |
| Research & Clinical Use | Reviews on medical imaging and ethics       | Aug–Oct 2025    |

---

##  Final Thoughts

The **latest breakthroughs in generative AI** reflect a dynamic ecosystem where increasingly sophisticated models are rapidly paired with industry-grade tools, infrastructure, and applications. From foundational LLMs like **GPT‑5** and **Qwen3-Max**, to viral visual models like **Nano Banana**, along with robust enterprise initiatives—AI is permeating both imaginative domains and practical workflows.

Crucially, with advancements come responsibilities: **security vulnerabilities** and **ethical concerns** are receiving fresh attention, while efforts like **veteran training** programs signal a strategic move to build workforce readiness.

If you'd like to dive deeper into any specific development—be it security, hardware, or a particular model—just let me know!

### Stateful Response

The OpenAI Responses API includes a stateful feature that enables continuity in interactions. 
By using the `response_id`, a conversation can persist across multiple queries, 
allowing users to refine or expand upon previous searches. This is particularly useful for iterative research, 
dynamic content generation, and applications that require follow-up queries based on prior responses.

In [16]:
fetched_response = client.responses.retrieve(response_id=web_search_response.id)
display(Markdown(fetched_response.output_text[:100]))

Below is a **thorough, structured overview** of the **latest developments in generative AI** as of *

### Continue Query with Web Search

In [17]:
continue_query = 'find different news'

continue_search_response = client.responses.create(
    model="gpt-4o",  # or another supported model
    input= continue_query,
    previous_response_id=web_search_response.id,
    tools=[
        {
            "type": "web_search"
        }
    ]
)

In [18]:
display(Markdown(continue_search_response.output_text))

Here’s a selection of **recent and notable generative AI developments (Nov 2025 timeframe)**, drawn from diverse sources and domains. Each update includes well-cited context and analysis.

---

##  OpenAI: Sora Video App Soars, Aardvark Security Agent Debuts

- **Sora Android Launch Success**  
  On November 6, 2025, OpenAI released *Sora*, its AI-powered video app, on Android. It instantly garnered around 470,000 downloads—surpassing its iOS debut by approximately 327%, which achieved around 360,000 downloads.([agiyes.com](https://www.agiyes.com/ainews/ai-news-from-november-1-7-2025/?utm_source=openai))

- **Aardvark: AI-Powered Security Researcher**  
  At the end of October, OpenAI unveiled *Aardvark*, an autonomous security researcher built on GPT‑5, now in private beta. With ~92% recall on seeded vulnerability tests, Aardvark has already uncovered 10 CVE-tracked security flaws.([aiwebbiz.com](https://aiwebbiz.com/blog/top-5-ai-news-of-the-week-november-2025/?utm_source=openai))  
  These milestones highlight OpenAI’s expansion into both the consumer AI space and advanced cybersecurity tooling.

---

##  Google: Multi-Agent Code Generation, New Models, and AI in Navigation

- **DS STAR: Autonomous Data Science Agent**  
  Google Research introduced *DS STAR*, a multi-agent system capable of transforming ambiguous business queries into executable Python code. It handles various input data formats—CSV, JSON, free text—enhancing automation in data workflows.([agiyes.com](https://www.agiyes.com/ainews/ai-news-from-november-1-7-2025/?utm_source=openai))

- **Gemini 3 Pro & Deep Research Enhancements**  
  A preview of *Gemini 3 Pro* surfaced on Vertex AI, expected to support massive 1-million-token context windows. Meanwhile, *Gemini AI* gained the *Deep Research* feature, which extracts information from Gmail, Drive, and Chat to autonomously generate in-depth reports.([agiyes.com](https://www.agiyes.com/ainews/ai-news-from-november-1-7-2025/?utm_source=openai))

- **Nano Banana 2 Coming Soon**  
  Google also hinted at *Nano Banana 2* (codename GEMPIX2), a new advanced image generation model expected to launch soon, building on the Nano Banana series.([agiyes.com](https://www.agiyes.com/ainews/ai-news-from-november-1-7-2025/?utm_source=openai))

- **Maps & Chrome Integration with AI**  
  *Gemini* will soon be embedded in Google Maps as a voice assistant, streamlining navigation by voice commands. Separately, Chrome (iOS/Android) now features an *AI Mode* button, simplifying access to AI-powered search functions; currently rolling out in the U.S. and expanding globally.([agiyes.com](https://www.agiyes.com/ainews/ai-news-from-november-1-7-2025/?utm_source=openai))  
  Together, these additions reveal a broad strategy by Google to entrench AI across core user experiences—from productivity to navigation.

---

##  Meta and HeyGen: AI Creativity Expands

- **Meta’s Vibes Platform Arrives in Europe**  
  Meta extended *Vibes*, its AI-generated short-video experience akin to TikTok or Reels, into Europe via the Meta AI app. The platform creates entirely AI-generated videos for users to view and produce.([agiyes.com](https://www.agiyes.com/ainews/ai-news-from-november-1-7-2025/?utm_source=openai))

- **HeyGen Launches AI Video Translator**  
  *HeyGen* released a hyper-realistic AI video translation tool. It produces localized video content where foreign speakers appear fluent—with synchronized lip movements, tone, and expressions. Available through web, iOS, and API, it also includes free trials.([agiyes.com](https://www.agiyes.com/ainews/ai-news-from-november-1-7-2025/?utm_source=openai))  
  These tools showcase momentum in generative AI enhancing and transforming multimedia communication.

---

##  Enterprise Tools & Workflow Enhancements: ClickUp 4.0

- **ClickUp 4.0 with AI Agents**  
  ClickUp’s version 4.0 arrives with a revamped UI and two embedded AI agents. These streamline project management, document collaboration, scheduling, messaging, and enterprise search—blending multiple work functions into a cohesive, AI-enhanced platform.([agiyes.com](https://www.agiyes.com/ainews/ai-news-from-november-1-7-2025/?utm_source=openai))  
  This reflects rising demand for integrated AI assistants in workplace productivity tools.

---

##  Business and Technology Insights: Projects and Ecosystem Trends

- **MIT Study: 95% of Enterprise GenAI Projects Failing**  
  A report from MIT, dated August 22, 2025, revealed that 95% of generative AI projects in businesses fail to deliver meaningful results—despite over $44 billion invested in the first half of 2025. Causes include inflated expectations, poor integration, and mismatched implementation. The study likens the trend to a software-age bubble.([timesofindia.indiatimes.com](https://timesofindia.indiatimes.com/technology/tech-news/mit-study-finds-95-of-generative-ai-projects-are-failing-only-hype-little-transformation/articleshow/123453071.cms?utm_source=openai))

- **Steam Gaming: Massive GenAI Adoption**  
  As of mid‑2025, 7,818 Steam games—about 7% of the entire library—disclosed use of generative AI, marking a 681% year-over-year rise. One in five games released in 2025 uses GenAI, primarily for visual assets (60%), followed by audio, narrative, marketing, and code. Some games even integrate AI at runtime for dynamic environments.([tomshardware.com](https://www.tomshardware.com/video-games/pc-gaming/1-in-5-steam-games-released-in-2025-use-generative-ai-up-nearly-700-percent-year-on-year-7-818-titles-disclose-genai-asset-usage-7-percent-of-entire-steam-library?utm_source=openai))

---

##  Executive Summary

| Sector            | Noteworthy Developments |
|------------------|--------------------------|
| **OpenAI**       | Sora app downloads surge; Aardvark security researcher launched |
| **Google**       | DS STAR code agent, Gemini 3 Pro preview, Deep Research, Nano Banana 2, AI in Maps and Chrome |
| **Meta / HeyGen**| Vibes short videos extend to Europe; AI video translator launched |
| **Enterprises**  | ClickUp 4.0 released, consolidating work tools via AI agents |
| **Research**     | MIT reports 95% of enterprise GenAI projects fail; Steam games show rapid GenAI adoption |

---

### Analysis & Insight

- **Consumer & Developer Innovation**: From *Sora* to *Gemini Deep Research*, AI is pushing into everyday tools—video, navigation, summarization, and complex code generation—indicating broad user and developer uptake.
- **AI-Powered Creativity and Localization**: Platforms like *Meta Vibes* and *HeyGen* highlight growing demand for AI-generated media and multilingual, lip-synced content creation.
- **Enterprise Ambivalence**: While tools like *ClickUp 4.0* champion AI integration, the MIT study warns that most business-level GenAI projects are faltering—suggesting growing pains in aligning AI with enterprise needs.
- **Emerging Cybersecurity Role**: OpenAI’s *Aardvark* signals AI’s expanding presence in proactive, autonomous security research.
- **Gaming and GenAI Intersection**: The surge of AI in gaming (Steam titles) hints at early generational shifts—raising new dialogues around transparency, creativity, and tech trust.

---

If you'd like deeper exploration into any of these topics—such as the technical underpinnings of Nano Banana 2, effectiveness of AI agents in enterprises, or the detailed implications of the MIT study—I'd be happy to dive further.

### Combining File Search and Web Search

This is an example of using file search to analyze private data and web search to retrieve public or the latest data. 
The Responses API allows developers to integrate these tools to enhance retrieval-augmented generation (RAG) applications. 
By combining file search with web search, users can leverage structured internal knowledge while also retrieving real-time 
information from external sources, ensuring comprehensive and up-to-date responses. 

In [19]:
combined_search_response = client.responses.create(
    model="gpt-4o",  # or another supported model
    input= query,
    temperature = 0,
    instructions="Retrieve the results from the file search first, and use the web search tool to expand the results with news resources",
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store_id],
    },
        {
            "type": "web_search"
        }
    ]
)

In [20]:
display(Markdown(combined_search_response.output_text))

Recent developments in generative AI include:

1. **Market Growth**: The global generative AI market is projected to reach $1.18 billion this year, highlighting its rapid expansion and the increasing demand for intelligent capabilities.

2. **Industry Applications**: Generative AI is being applied across various sectors, from content generation to design systems, reshaping how businesses innovate.

3. **Technological Advancements**: New tools like OpenAI's Sora2 are enabling the creation of cinematic videos from simple prompts, showcasing the potential for creativity and innovation.

4. **Business Integration**: Companies like Watsonx are integrating generative AI into enterprise solutions, allowing for the development of custom large language models to enhance customer engagement and streamline processes.

5. **Ethical and Regulatory Considerations**: There is ongoing discussion about the regulation and ethics of generative AI, focusing on issues like deepfakes and content ownership.

These developments indicate a significant impact on various industries, with ongoing discussions about the ethical implications and potential of generative AI.

# 🧩 Try It Yourself: Two-Step RAG (Private Data + Combined Search)

## Step 1 — Upload & Create Vector Store
1. Upload a short text file (e.g., `my_notes.txt`) to your notebook instance.  
2. Create a **vector store** and **ingest** your uploaded file.  
3. Run a simple test query to verify retrieval:  

In [21]:
with open('Reflection 9.txt', 'rb') as f:
    file = client.files.create(
        file=f,            # file-like object
        purpose="assistants"
    )

file_id = file.id
print(file_id)

file-HekfAQbcUz6cymma4e9rsG


In [22]:
attach_status =client.vector_stores.files.create(
    vector_store_id=vector_store_id,
    file_id=file_id
            )

print(attach_status.id)

file-HekfAQbcUz6cymma4e9rsG


In [24]:
query = "opinions on Odysseus"

In [26]:
search_results = client.vector_stores.search(
    vector_store_id=vector_store_id,
    query=query
)

for result in search_results.data[:5]:
    print(result.content[0].text[:100] + '\n Relevant score: ' + str(result.score))

I believe that the lessons gained from the Odyssey are equally valuable as the ones taught in the Il
 Relevant score: 0.8383876847179378
co/xvcgwvUZC8 #MachineLearning #AdversarialAI #SmartAI #AI #EnterpriseAI #GenerativeAI #GenAI #MLAlg
 Relevant score: 0.048011566226767756
They have been VERY clear that Chatgpt, these AI videos…"
  }
},
{
  "_id": {
    "$oid": "68e56b1e1
 Relevant score: 0.040687592554431364
"tweet": {
    "text": "RT @AmandaDannielle: People keep telling y'all which types of AI to stay awa
 Relevant score: 0.04060798878832805
They have been VERY clear that Chatgpt, these AI videos…"
  }
},
{
  "_id": {
    "$oid": "68e56b1f1
 Relevant score: 0.03078232370915975


## Step 2 — Combine File Search with Web Search
1. Enable both **file_search** and **web_search** in the Responses API.  
2. Use a prompt that asks the model to merge insights from both sources.  
   > Example: “Using my uploaded notes and the latest web information, summarize the current trends on this topic.”  
3. Review how the answer from your file and **current info** from the web.

✅ You’ve created a RAG system that combines **private** and **public** data for comprehensive, up-to-date analysis.


In [27]:
file_search_response = client.responses.create(
    input= query,
    model="gpt-4o",
    temperature = 0,
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store_id],
    }]
)

In [28]:
display(Markdown(file_search_response.output_text))


The opinion on Odysseus from the document suggests that while the author wasn't a huge fan of Odysseus as a protagonist, they found his journey and the emotional depth of his story in "The Odyssey" to be impactful and memorable. The narrative of Odysseus' struggles and transformations was seen as more engaging than the themes of rage and pride in "The Iliad." The author appreciated the philosophical aspects of "The Odyssey" but preferred Achilles as a more relatable protagonist.

In [29]:
web_search_response = client.responses.create(
    model="gpt-4o",  # or another supported model
    input= query,
    tools=[
        {
            "type": "web_search"
        }
    ]
)

In [31]:
display(Markdown(web_search_response.output_text))

Odysseus is a complex character from Greek mythology, particularly known for his role in Homer's epics, "The Iliad" and "The Odyssey." Opinions about him vary widely:

1. **Cunning and Intelligent**: Odysseus is often admired for his wit and intelligence. His cleverness is seen in the creation of the Trojan Horse, which led to the fall of Troy.

2. **Flawed Hero**: Despite his strengths, he is also flawed. His hubris and lack of foresight sometimes lead to unnecessary conflicts and suffering, such as when he taunts the Cyclops Polyphemus.

3. **Enduring and Resilient**: Odysseus is celebrated for his endurance and resilience. His ten-year journey home to Ithaca after the Trojan War is filled with numerous trials, testing his resolve and resourcefulness.

4. **Loyalty and Fidelity**: His commitment to return to his wife, Penelope, is seen as a testament to his loyalty, though this is complicated by his interactions with figures like Circe and Calypso.

5. **Moral Ambiguity**: Modern interpretations sometimes focus on the moral complexities of his character, examining his decisions and their ethical implications.

Overall, Odysseus is seen as a heroic yet deeply human figure, embodying both admirable traits and significant flaws.

In [32]:
combined_search_response = client.responses.create(
    model="gpt-4o",  # or another supported model
    input= query,
    temperature = 0,
    instructions="Using the uploaded notes and information online, summarize general opinions on this topic",
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store_id],
    },
        {
            "type": "web_search"
        }
    ]
)

In [33]:
display(Markdown(combined_search_response.output_text))

Opinions on Odysseus are varied and often reflect personal preferences for different aspects of his character and story. In the provided notes, one perspective suggests that while Odysseus is not favored as a protagonist compared to Achilles, his journey and the emotional depth of his experiences are compelling. The Odyssey is appreciated for its philosophical depth, exploring themes of identity, struggle, and transformation through Odysseus' trials. This contrasts with the Iliad, which is seen as more focused on themes of rage and pride.